# OpenPIE-0.6 checkpoint reconstruction evidence

Run this notebook on Kaggle with internet enabled. It downloads the OpenPIE-0.6 checkpoint, exports tensor names and shapes without loading tensor data, clones the related OpenPI source, and packages only small metadata files for download.

**Expected downloads:** approximately 15 GB for the policy and value checkpoints. Do not upload the `.safetensors` files back to the repository.

## 1. Install inspection dependencies

In [ ]:
!pip -q install huggingface_hub safetensors

## 2. Configure Kaggle paths

The checkpoint is kept outside the repository checkout. Only metadata outputs are copied into the output bundle.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess

WORK = Path('/kaggle/working')
CHECKPOINT_DIR = WORK / 'openpie-0.6'
SOURCE_DIR = WORK / 'openpi-source'
OUTPUT_DIR = WORK / 'openpie-reconstruction-output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Checkpoint:', CHECKPOINT_DIR)
print('Output:', OUTPUT_DIR)


## 3. Check Kaggle internet access

If this cell cannot resolve `huggingface.co`, enable **Internet** in the Kaggle notebook sidebar, restart the session, and rerun from this cell.

In [ ]:
import socket
import urllib.request

try:
    print('DNS:', socket.gethostbyname('huggingface.co'))
    with urllib.request.urlopen('https://huggingface.co', timeout=20) as response:
        print('HTTPS:', response.status)
except Exception as exc:
    raise RuntimeError(
        'Kaggle cannot reach Hugging Face. Enable Notebook Internet in the Kaggle sidebar, restart the session, and rerun this cell.'
    ) from exc

## 4. Download the OpenPIE-0.6 checkpoint

This downloads the three published files from Hugging Face. Reruns resume files already present in the Hub cache.

In [ ]:
import time
from huggingface_hub import snapshot_download

for attempt in range(5):
    try:
        snapshot_download(
            repo_id='exla-ai/openpie-0.6',
            local_dir=str(CHECKPOINT_DIR),
            allow_patterns=['README.md', 'config.json', 'policy.safetensors', 'value_fn.safetensors'],
            max_workers=2,
        )
        break
    except Exception as exc:
        if attempt == 4:
            raise RuntimeError('OpenPIE download failed after 5 attempts. Check Kaggle Internet and storage.') from exc
        wait_seconds = 10 * (attempt + 1)
        print(f'Download attempt {attempt + 1} failed: {exc}')
        print(f'Retrying in {wait_seconds} seconds...')
        time.sleep(wait_seconds)
print('Downloaded:', sorted(p.name for p in CHECKPOINT_DIR.iterdir()))

## 4. Read and validate the published configuration

In [ ]:
config = json.loads((CHECKPOINT_DIR / 'config.json').read_text())
print(json.dumps(config, indent=2))
expected = {'action_dim': 14, 'state_dim': 14, 'action_horizon': 50}
for key, value in expected.items():
    assert config.get(key) == value, (key, config.get(key), value)
print('Published OpenPIE-0.6 dimensions validated.')

## 5. Export tensor names and shapes

This uses `safe_open`, so tensor values are not loaded into RAM.

In [ ]:
from safetensors import safe_open
def export_tensor_metadata(checkpoint_path, output_path):
    entries = []
    parameter_count = 0
    with safe_open(str(checkpoint_path), framework='pt', device='cpu') as handle:
        for name in handle.keys():
            shape = list(handle.get_slice(name).get_shape())
            count = 1
            for dimension in shape:
                count *= dimension
            parameter_count += count
            entries.append({'name': name, 'shape': shape, 'parameters': count})
    result = {
        'checkpoint': str(checkpoint_path),
        'tensor_count': len(entries),
        'parameter_count': parameter_count,
        'parameters_billions': parameter_count / 1e9,
        'tensors': entries,
    }
    output_path.write_text(json.dumps(result, indent=2) + '\n')
    return result

policy_metadata = export_tensor_metadata(
    CHECKPOINT_DIR / 'policy.safetensors', OUTPUT_DIR / 'policy_keys.json'
)
value_metadata = export_tensor_metadata(
    CHECKPOINT_DIR / 'value_fn.safetensors', OUTPUT_DIR / 'value_fn_keys.json'
)
print('Policy:', policy_metadata['tensor_count'], 'tensors;', policy_metadata['parameters_billions'], 'B params')
print('Value:', value_metadata['tensor_count'], 'tensors;', value_metadata['parameters_billions'], 'B params')

## 6. Print reconstruction-relevant tensor shapes

In [ ]:
markers = ('action_in_proj', 'action_out_proj', 'time_mlp', 'state', 'gripper', 'paligemma', 'expert')
selected = [
    entry for entry in policy_metadata['tensors']
    if any(marker in entry['name'] for marker in markers)
]
for entry in selected:
    print(entry['name'], tuple(entry['shape']))

## 7. Clone and scan related OpenPI source

This is a reference implementation, not proof that its architecture exactly matches OpenPIE-0.6.

In [ ]:
if not SOURCE_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--filter=blob:none', '--no-checkout',
         'https://github.com/Physical-Intelligence/openpi.git', str(SOURCE_DIR)],
        check=True,
    )
    subprocess.run(['git', '-C', str(SOURCE_DIR), 'sparse-checkout', 'set', 'openpi', 'scripts', 'packages'], check=True)
    subprocess.run(['git', '-C', str(SOURCE_DIR), 'checkout'], check=True)
print('OpenPI source clone ready without third-party submodules.')
markers = ('action_in_proj', 'action_out_proj', 'PaliGemma', 'paligemma', 'action_expert', 'flow')
source_matches = []
for path in SOURCE_DIR.rglob('*.py'):
    try:
        text = path.read_text(encoding='utf-8', errors='ignore')
    except OSError:
        continue
    found = [marker for marker in markers if marker in text]
    if found:
        source_matches.append({'path': str(path), 'markers': found})
print('Candidate source files:', len(source_matches))
for item in source_matches[:100]:
    print(item['path'], item['markers'])

## 8. Write a reconstruction report

In [ ]:
report = {
    'model_repo': 'exla-ai/openpie-0.6',
    'config': config,
    'policy_tensor_count': policy_metadata['tensor_count'],
    'policy_parameter_count': policy_metadata['parameter_count'],
    'value_tensor_count': value_metadata['tensor_count'],
    'value_parameter_count': value_metadata['parameter_count'],
    'related_source_repo': 'https://github.com/Physical-Intelligence/openpi',
    'related_source_matches': source_matches,
    'warning': 'Related OpenPI source is not confirmed checkpoint-compatible with OpenPIE-0.6.',
    'next_step': 'Compare checkpoint key hierarchy and shapes with candidate model modules before implementing a forward pass.',
}
(OUTPUT_DIR / 'reconstruction_report.json').write_text(json.dumps(report, indent=2))
print(json.dumps({k: v for k, v in report.items() if k != 'related_source_matches'}, indent=2))

## 9. Package only the small outputs

Download `openpie-reconstruction-metadata.zip` from the Kaggle output panel and provide its JSON files for architecture comparison.

In [ ]:
archive_base = WORK / 'openpie-reconstruction-metadata'
archive_path = shutil.make_archive(str(archive_base), 'zip', OUTPUT_DIR)
print('Created:', archive_path)
print('Files:', sorted(p.name for p in OUTPUT_DIR.iterdir()))

## 10. Prepare the PyTorch runtime

Run this before loading the model. Do not delete `openpi-source` until inference is complete.

In [ ]:
import importlib
import subprocess
import sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'jaxtyping==0.2.36', 'safetensors', 'sentencepiece'
], check=True)
print('Runtime dependencies installed.')
print('CUDA available:', importlib.import_module('torch').cuda.is_available())

## 11. Load the reconstructed policy

Upload `model_config.py` and `load_policy.py` from this folder as Kaggle input, or place them in `/kaggle/working/openpie-0.6`.

In [ ]:
import sys
from pathlib import Path

LOCAL_CODE = Path('/kaggle/working/openpie-0.6')
if not (LOCAL_CODE / 'load_policy.py').is_file():
    raise FileNotFoundError(
        'Upload openpie-0.6/load_policy.py and model_config.py before running this cell.'
    )
sys.path.insert(0, str(LOCAL_CODE))
from load_policy import load_policy

import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cpu':
    raise RuntimeError('A GPU runtime is required for the inference cell.')
model, load_report = load_policy(
    POLICY_PATH,
    SOURCE_DIR,
    device='cpu',
    strict=True,
)
model = model.to(device=device, dtype=torch.bfloat16)
print(load_report)
print('Loaded device:', device)


## 12. Run a synthetic shape-and-finiteness check

This checks reconstruction and post-inference arm selection only. It does not prove that the dummy observation is meaningful or that the actions are safe for CRISP.

In [ ]:
from types import SimpleNamespace
import numpy as np

batch_size = 1
dummy_images = {
    'base_0_rgb': torch.zeros((batch_size, 3, 224, 224), device=device, dtype=torch.float32),
    'left_wrist_0_rgb': torch.zeros((batch_size, 3, 224, 224), device=device, dtype=torch.float32),
    'right_wrist_0_rgb': torch.zeros((batch_size, 3, 224, 224), device=device, dtype=torch.float32),
}
+    observation = SimpleNamespace(
    images=dummy_images,
    image_masks={key: torch.ones((batch_size,), device=device, dtype=torch.bool) for key in dummy_images},
    state=torch.zeros((batch_size, 14), device=device, dtype=torch.float32),
    tokenized_prompt=torch.zeros((batch_size, 200), device=device, dtype=torch.long),
    tokenized_prompt_mask=torch.ones((batch_size, 200), device=device, dtype=torch.bool),
)
+    with torch.inference_mode():
    actions = model.sample_actions(device, observation, num_steps=1)
actions_np = actions[0].float().cpu().numpy()
sys.path.insert(0, str(LOCAL_CODE))
from openpie_14d_to_7d import select_arm_action, validate_action_chunk
validated = validate_action_chunk(actions_np, expected_horizon=50)
selected = select_arm_action(validated, arm='left')
assert selected.shape == (50, 7)
assert np.isfinite(selected).all()
np.save(OUTPUT_DIR / 'synthetic_actions_14d.npy', validated)
np.save(OUTPUT_DIR / 'synthetic_left_actions_7d.npy', selected)
print('14D:', validated.shape, '7D:', selected.shape)
print('14D range:', float(validated.min()), float(validated.max()))

## 13. Package outputs after inference

Run the archive cell again after the inference cell so the action arrays are included.

In [ ]:
archive_base = WORK / 'openpie-reconstruction-with-inference'
archive_path = shutil.make_archive(str(archive_base), 'zip', OUTPUT_DIR)
print('Created:', archive_path)
print('Files:', sorted(p.name for p in OUTPUT_DIR.iterdir()))